In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from credit_default.data import load_credit_data
from credit_default.features import clean_credit_data, split_features_target, get_feature_groups

from sklearn.model_selection import train_test_split

from credit_default.data import load_credit_data
from credit_default.features import (
    clean_credit_data,
    split_features_target,
    get_feature_groups,
    add_credit_behaviour_features,
    check_for_vif,
    split_pos_neg_features,
    skew_checker
)

from credit_default.modeling.pipelines import build_logistic_regression_pipeline, build_logistic_regression_pipeline_np1log

from credit_default.evaluation import (
    evaluate_classifier,
    threshold_report,
    add_business_utility
)


In [2]:
raw_data = load_credit_data()
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

In [3]:
feature_groups = get_feature_groups()
feature_groups

FeatureGroups(categorical_cols=['SEX', 'EDUCATION', 'MARRIAGE_CLEAN', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'], numeric_cols=['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], bill_cols=['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'], pay_amount_cols=['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], payment_status_cols=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'])

In [4]:
clean_data = clean_credit_data(raw_data)
clean_data

,LIMIT_BAL,SEX,EDUCATION,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month,MARRIAGE_CLEAN
0,20000,2,2,24,2,2,-1,-1,-2,-2,...,0,0,0,689,0,0,0,0,1,1
1,120000,2,2,26,-1,2,0,0,0,2,...,3455,3261,0,1000,1000,1000,0,2000,1,2
2,90000,2,2,34,0,0,0,0,0,0,...,14948,15549,1518,1500,1000,1000,1000,5000,0,2
3,50000,2,2,37,0,0,0,0,0,0,...,28959,29547,2000,2019,1200,1100,1069,1000,0,1
4,50000,1,2,57,-1,0,-1,0,0,0,...,19146,19131,2000,36681,10000,9000,689,679,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,39,0,0,0,0,0,0,...,31237,15980,8500,20000,5003,3047,5000,1000,0,1
29996,150000,1,3,43,-1,-1,-1,-1,0,0,...,5190,0,1837,3526,8998,129,0,0,0,2
29997,30000,1,2,37,4,3,2,-1,0,0,...,20582,19357,0,0,22000,4200,2000,3100,1,2
29998,80000,1,3,41,1,-1,0,0,0,-1,...,11855,48944,85900,3409,1178,1926,52964,1804,1,1


In [5]:
fe_clean_data, columnas_anadidas = add_credit_behaviour_features(clean_data) ## añade FE de mean std max min
fe_clean_data

,LIMIT_BAL,SEX,EDUCATION,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,default payment next month,MARRIAGE_CLEAN,BILL_AMT_mean,BILL_AMT_max,BILL_AMT_std,PAY_AMT_mean,PAY_AMT_max,PAY_AMT_std,debt_to_limit,payment_to_debt
0,20000,2,2,24,2,2,-1,-1,-2,-2,...,1,1,1284.000000,3913,1761.633219,114.833333,689,281.283072,0.064200,0.089434
1,120000,2,2,26,-1,2,0,0,0,2,...,1,2,2846.166667,3455,637.967841,833.333333,2000,752.772653,0.023718,0.292791
2,90000,2,2,34,0,0,0,0,0,0,...,0,2,16942.166667,29239,6064.518593,1836.333333,5000,1569.815488,0.188246,0.108388
3,50000,2,2,37,0,0,0,0,0,0,...,0,1,38555.666667,49291,10565.793518,1398.000000,2019,478.058155,0.771113,0.036259
4,50000,1,2,57,-1,0,-1,0,0,0,...,0,1,18223.166667,35835,10668.590074,9841.500000,36681,13786.230736,0.364463,0.540054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,39,0,0,0,0,0,0,...,0,1,120891.500000,208365,86697.439530,7091.666667,20000,6794.318234,0.549507,0.058661
29996,150000,1,3,43,-1,-1,-1,-1,0,0,...,0,2,3530.333333,8979,3200.534247,2415.000000,8998,3515.523859,0.023536,0.684071
29997,30000,1,2,37,4,3,2,-1,0,0,...,1,2,11749.333333,20878,9354.149660,5216.666667,22000,8390.093365,0.391644,0.443997
29998,80000,1,3,41,1,-1,0,0,0,-1,...,1,1,44435.166667,78379,32992.487323,24530.166667,85900,36314.167188,0.555440,0.552044


In [6]:
X, y = split_features_target(fe_clean_data)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

X_train_fe_numeric_cols = feature_groups.numeric_cols.copy()
X_train_fe_numeric_cols.extend(columnas_anadidas)

X_train_og_numeric_cols = feature_groups.numeric_cols.copy()

## VIF Check

In [7]:
vif_data = check_for_vif(X_train, X_train_og_numeric_cols)
## VIF from original variables
vif_data

,variable,VIF
4,BILL_AMT2,26.703983
7,BILL_AMT5,24.378164
5,BILL_AMT3,23.215940
6,BILL_AMT4,20.229409
8,BILL_AMT6,14.522453
3,BILL_AMT1,14.326928
10,PAY_AMT2,2.348247
13,PAY_AMT5,1.725191
11,PAY_AMT3,1.635812
9,PAY_AMT1,1.627797


In [8]:
vif_data =check_for_vif(X_train, X_train_fe_numeric_cols)
vif_data

/Users/marcoantoniogarciamartinez/Documents/Python Local/Entorno_DS/Python_Versions/3.11.15/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variable,VIF
12,PAY_AMT4,inf
9,PAY_AMT1,inf
18,PAY_AMT_mean,inf
15,BILL_AMT_mean,inf
14,PAY_AMT6,inf
13,PAY_AMT5,inf
11,PAY_AMT3,inf
10,PAY_AMT2,inf
8,BILL_AMT6,inf
7,BILL_AMT5,inf


## Removal

At this point I decided to continue with FE cols 

In [9]:
pay_to_remove = [f'PAY_AMT{i}' for i in range(1, 7)]

for variable in pay_to_remove:
    X_train_fe_numeric_cols.remove(variable)

vif_data = check_for_vif(X_train, X_train_fe_numeric_cols)
vif_data

/Users/marcoantoniogarciamartinez/Documents/Python Local/Entorno_DS/Python_Versions/3.11.15/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variable,VIF
3,BILL_AMT1,inf
4,BILL_AMT2,inf
5,BILL_AMT3,inf
6,BILL_AMT4,inf
7,BILL_AMT5,inf
8,BILL_AMT6,inf
9,BILL_AMT_mean,inf
13,PAY_AMT_max,213.259128
14,PAY_AMT_std,184.857055
10,BILL_AMT_max,79.237089


In [10]:
pay_to_remove = [f'BILL_AMT{i}' for i in range(1, 7)]

for variable in pay_to_remove:
    X_train_fe_numeric_cols.remove(variable)

vif_data = check_for_vif(X_train, X_train_fe_numeric_cols)
vif_data

,variable,VIF
7,PAY_AMT_max,211.752366
8,PAY_AMT_std,183.685960
4,BILL_AMT_max,59.982498
3,BILL_AMT_mean,38.841397
5,BILL_AMT_std,8.757625
6,PAY_AMT_mean,8.501129
9,debt_to_limit,2.719439
1,LIMIT_BAL,2.238838
2,AGE,1.021757
10,payment_to_debt,1.021426


In [11]:
candidatas_preliminares = vif_data[vif_data['VIF'] <= 10]['variable'].tolist()

In [12]:
vif_data = check_for_vif(X_train, candidatas_preliminares)
vif_data

,variable,VIF
2,PAY_AMT_mean,1.675962
1,BILL_AMT_std,1.657435
4,LIMIT_BAL,1.492091
3,debt_to_limit,1.304490
5,AGE,1.021296
6,payment_to_debt,1.017432


In [13]:
exporter_candidatos = candidatas_preliminares.copy()

for variable in feature_groups.categorical_cols:
    exporter_candidatos.append(variable) ## reintegramos categoricals

train_candidate_frame = X_train[exporter_candidatos]


In [14]:
## Este modelo recibe ya las columnas deflatadas
model = build_logistic_regression_pipeline(
    numeric_cols = candidatas_preliminares,
    categorical_cols =feature_groups.categorical_cols
)
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [15]:
numeric_columnas =  candidatas_preliminares.copy()
columns_no_skewed, columns_yes_skewed = skew_checker(X_train, numeric_columnas)
neg_no_log, pos_yes_log = split_pos_neg_features(X_train, columns_yes_skewed)

In [ ]:
model_b = build_logistic_regression_pipeline_np1log(
    cols_to_log1p = pos_yes_log,
    cols_to_numeric = columns_no_skewed,
    cols_to_yeo = neg_no_log,
    categorical_cols =feature_groups.categorical_cols 
)

In [17]:
model_b.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log1p_num', ...), ('yeo', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [18]:
y_proba = model_b.predict_proba(X_test)[:, 1]

evaluate_classifier(
    y_true = y_test,
    y_proba = y_proba,
    threshold = 0.5
    
    )

{'threshold': 0.5,
 'accuracy': 0.7682560853617872,
 'precision': 0.4808277541083384,
 'recall': 0.5953278070836473,
 'f1': 0.531986531986532,
 'roc_auc': 0.7680649752993386,
 'pr_auc': 0.5410250662154319,
 'tn': 3818,
 'fp': 853,
 'fn': 537,
 'tp': 790}